# Regressão Logística: Previsão de Cancelamento de Clientes (Churn)

**Atividade 7 — Trabalho em grupo**

Este notebook reproduz, em formato executável, a análise apresentada no relatório em LaTeX, cobrindo todos os itens pedidos na atividade: descrição do problema, descrição e tratamento dos dados, análise exploratória, divisão treino/teste, ajuste do modelo de regressão logística, interpretação de coeficientes e razões de chance, avaliação de significância, Curva ROC/AUC, matriz de confusão e conclusões.


## 1. Introdução e Objetivo

O cancelamento de clientes, conhecido no meio empresarial como *churn*, é um dos principais problemas enfrentados por empresas que possuem serviços de assinatura, como operadoras de telecomunicações. Reter um cliente já existente costuma ser mais barato do que conquistar um novo, de modo que entender quais fatores levam um cliente a cancelar o serviço tem valor direto para a tomada de decisão da empresa.

Este trabalho utiliza o conjunto de dados **Telco Customer Churn** (Kaggle), composto por 7043 clientes de uma empresa de telecomunicações, com informações cadastrais, serviços contratados, forma de pagamento e status de cancelamento.

**Objetivo:** ajustar um modelo de regressão logística para estimar a probabilidade de um cliente cancelar o serviço (*churn*), a partir de suas características cadastrais, dos serviços contratados e das condições do contrato. Pretende-se também avaliar quais covariáveis são significativas, interpretar seus efeitos por meio das razões de chance (*odds ratios*), e medir a capacidade preditiva do modelo em dados não usados na sua construção (amostra de teste).


In [ ]:
# Pacotes utilizados ao longo de toda a análise
library(dplyr)      # manipulação de dados
library(ggplot2)     # gráficos
library(caret)        # particionamento treino/teste e matriz de confusão
library(pROC)          # curva ROC e AUC

set.seed(123) # reprodutibilidade (treino/teste)


## 2. Descrição do Conjunto de Dados

O banco de dados original possui 7043 linhas e 21 colunas.

**Variável resposta:** `Churn`, que indica se o cliente cancelou o serviço (`Yes`) ou permaneceu com a empresa (`No`). Será recodificada como binária numérica (1 = cancelou, 0 = não cancelou).

**Covariáveis utilizadas:**

- `gender`, `SeniorCitizen`, `Partner`, `Dependents` — características cadastrais;
- `tenure` — número de meses que o cliente permanece na empresa;
- `PhoneService`, `MultipleLines` — serviço de telefonia;
- `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` — serviço de internet e complementos;
- `Contract`, `PaperlessBilling`, `PaymentMethod` — condições contratuais;
- `MonthlyCharges`, `TotalCharges` — valores cobrados.

A coluna `customerID` foi removida por se tratar apenas de um identificador do cliente, sem relação com a chance de cancelamento.


In [ ]:
# Leitura dos dados
dados <- read.csv("WA_Fn-UseC_-Telco-Customer-Churn.csv", stringsAsFactors = FALSE)

head(dados)


In [ ]:
dim(dados)   # número de linhas e colunas
str(dados)   # tipos das variáveis


### Recodificação da variável resposta

A variável `Churn` é recodificada para 1 (cancelou) e 0 (não cancelou), e sua distribuição é verificada.


In [ ]:
dados$Churn <- ifelse(dados$Churn == "Yes", 1, 0)

table(dados$Churn)
prop.table(table(dados$Churn))


Do total de 7043 clientes, 5174 (73,5%) não cancelaram o serviço e 1869 (26,5%) cancelaram. Trata-se, portanto, de uma base moderadamente desbalanceada — ponto que será retomado na divisão da amostra e na escolha do ponto de corte do modelo.


In [ ]:
# Remoção do identificador do cliente (não é covariável)
dados$customerID <- NULL


### Tratamento dos dados

Ao importar o banco, a coluna `TotalCharges` apresenta valores faltantes. Investigamos essas linhas antes de decidir como tratá-las.


In [ ]:
sum(is.na(dados$TotalCharges))

dados[is.na(dados$TotalCharges), c("tenure", "MonthlyCharges", "TotalCharges")]


Todas as 11 linhas com `TotalCharges` faltante correspondem a clientes com `tenure` igual a zero, ou seja, clientes que acabaram de contratar o serviço e ainda não completaram um mês de cobrança. Como o valor faltante é, na prática, implicado pela própria variável `tenure` (cobrança total ainda seria zero), optamos por **imputar o valor 0** nessas observações, em vez de removê-las, preservando a base completa de 7043 clientes.


In [ ]:
dados$TotalCharges[is.na(dados$TotalCharges)] <- 0

sum(is.na(dados)) # confirma que não há mais valores faltantes no banco


In [ ]:
# Conversão das variáveis de texto para fator (variáveis categóricas)
dados <- dados %>% mutate(across(where(is.character), as.factor))

# SeniorCitizen estava codificada como 0/1; tratamos como fator com rótulos
dados$SeniorCitizen <- factor(dados$SeniorCitizen, labels = c("No", "Yes"))

str(dados)


## 3. Análise Exploratória

### Distribuição da variável resposta


In [ ]:
ggplot(dados, aes(x = factor(Churn, labels = c("Nao cancelou", "Cancelou")))) +
  geom_bar(fill = "steelblue") +
  labs(title = "Distribuicao da variavel resposta (Churn)",
       x = "", y = "Numero de clientes")


### Covariáveis numéricas

Resumo estatístico do tempo de contrato (`tenure`), da mensalidade (`MonthlyCharges`) e do total cobrado (`TotalCharges`).


In [ ]:
summary(dados[, c("tenure", "MonthlyCharges", "TotalCharges")])


### Tempo de contrato e mensalidade por status de churn

Comparamos a distribuição de `tenure` e `MonthlyCharges` entre clientes que cancelaram e clientes que permaneceram com a empresa.


In [ ]:
ggplot(dados, aes(x = factor(Churn, labels = c("Nao cancelou", "Cancelou")),
                  y = tenure)) +
  geom_boxplot(fill = "lightblue") +
  labs(title = "Tempo de contrato (meses) por status de churn",
       x = "", y = "Tenure (meses)")


In [ ]:
ggplot(dados, aes(x = factor(Churn, labels = c("Nao cancelou", "Cancelou")),
                  y = MonthlyCharges)) +
  geom_boxplot(fill = "lightgreen") +
  labs(title = "Mensalidade por status de churn",
       x = "", y = "Mensalidade (MonthlyCharges)")


Clientes que cancelaram tendem a ter um tempo de contrato menor — o que é esperado, já que clientes mais antigos já demonstraram algum nível de fidelidade ao serviço — e tendem a pagar mensalidades mais altas.

### Covariáveis categóricas: proporção de churn por categoria


In [ ]:
prop.table(table(dados$Contract, dados$Churn), margin = 1)

ggplot(dados, aes(x = Contract, fill = factor(Churn, labels = c("Nao cancelou", "Cancelou")))) +
  geom_bar(position = "fill") +
  labs(title = "Proporcao de churn por tipo de contrato",
       x = "Tipo de contrato", y = "Proporcao", fill = "Churn")


Clientes com contrato mensal (*Month-to-month*) cancelam em uma proporção bem maior (42,7%) do que clientes com contrato de um ano (11,3%) ou de dois anos (2,8%), sugerindo que essa covariável deve ter papel importante no modelo.


In [ ]:
prop.table(table(dados$PaymentMethod, dados$Churn), margin = 1)

ggplot(dados, aes(x = PaymentMethod, fill = factor(Churn, labels = c("Nao cancelou", "Cancelou")))) +
  geom_bar(position = "fill") +
  labs(title = "Proporcao de churn por metodo de pagamento",
       x = "Metodo de pagamento", y = "Proporcao", fill = "Churn") +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))


Clientes que pagam por cheque eletrônico (*Electronic check*) apresentam proporção de cancelamento consideravelmente maior do que os demais métodos, próxima de 45%.


## 4. Divisão da Amostra (Treino e Teste)

Como a variável resposta é desbalanceada (73,5% x 26,5%), a divisão entre treino e teste é feita de forma **estratificada**, e não por amostragem aleatória simples. A estratificação garante que a mesma proporção de cancelamento seja preservada tanto na amostra de treino quanto na amostra de teste, evitando que uma divisão aleatória concentre, por acaso, mais casos de churn em um dos dois conjuntos.

Foi adotada a proporção de 70% dos dados para treino e 30% para teste.


In [ ]:
indices_treino <- createDataPartition(dados$Churn, p = 0.7, list = FALSE)

treino <- dados[indices_treino, ]
teste  <- dados[-indices_treino, ]

nrow(treino)
nrow(teste)

# Verificando que a proporcao de churn foi preservada em ambas as amostras
prop.table(table(treino$Churn))
prop.table(table(teste$Churn))


A divisão resultou em 4931 observações na amostra de treino e 2112 observações na amostra de teste, com a proporção de cancelamento preservada em ambos os conjuntos.


## 5. Ajuste do Modelo de Regressão Logística

O modelo é ajustado **exclusivamente na amostra de treino**, utilizando todas as covariáveis disponíveis como ponto de partida (modelo completo).


In [ ]:
modelo_completo <- glm(Churn ~ ., data = treino, family = binomial(link = "logit"))

summary(modelo_completo)


### Seleção de variáveis

A partir do modelo completo, aplicamos uma seleção de variáveis por critério de informação (`stepAIC`, método *stepwise* em ambas as direções) para obter um modelo mais parcimonioso, eliminando covariáveis que não contribuem para o ajuste.


In [ ]:
library(MASS)

modelo_final <- stepAIC(modelo_completo, direction = "both", trace = FALSE)

summary(modelo_final)


## 6. Interpretação dos Coeficientes e Razões de Chance (Odds Ratios)

Os coeficientes do modelo logístico são interpretados na escala do log-odds. Para facilitar a interpretação, calculamos as razões de chance (**odds ratios**), obtidas por $e^{\hat{\beta}}$, junto com os respectivos intervalos de confiança de 95%.

- OR > 1: a covariável **aumenta** a chance de cancelamento;
- OR < 1: a covariável **diminui** a chance de cancelamento;
- OR = 1: a covariável não tem efeito sobre a chance de cancelamento.


In [ ]:
# Razoes de chance (odds ratios) e intervalos de confianca de 95%
odds_ratios <- exp(cbind(OR = coef(modelo_final), confint(modelo_final)))

round(odds_ratios, 3)


**Como interpretar (exemplos a preencher com os resultados obtidos):**

- Para variáveis contínuas como `tenure`, o OR representa a variação multiplicativa na chance de churn a cada mês adicional de permanência (espera-se OR < 1, indicando efeito protetor).
- Para variáveis categóricas como `Contract`, o OR de cada categoria é relativo à categoria de referência (`Month-to-month`), sendo esperado OR bem menor que 1 para contratos de um e dois anos, confirmando o padrão observado na análise exploratória.
- Para `MonthlyCharges`, espera-se OR > 1, refletindo a associação entre mensalidades mais altas e maior propensão ao cancelamento observada na Seção 3.

*Observação: os comentários acima devem ser ajustados pelo grupo de acordo com os valores numéricos efetivamente obtidos na execução do código.*


## 7. Avaliação da Significância das Covariáveis e Justificativa do Modelo Final

Além dos testes de Wald individuais reportados em `summary(modelo_final)` (coluna `Pr(>|z|)`), comparamos o modelo completo e o modelo reduzido por meio do teste da razão de verossimilhanças (*Likelihood Ratio Test*), e avaliamos a qualidade do ajuste através do AIC.


In [ ]:
# Teste da razao de verossimilhancas entre o modelo completo e o modelo final (reduzido)
anova(modelo_final, modelo_completo, test = "Chisq")

# Comparacao do AIC dos dois modelos
AIC(modelo_completo, modelo_final)


Se o teste da razão de verossimilhanças não indicar diferença significativa entre os dois modelos (p-valor alto), e o modelo reduzido apresentar AIC igual ou menor, opta-se pelo **modelo final (reduzido)**, por ser mais parcimonioso, mantendo apenas as covariáveis estatisticamente significativas e/ou relevantes para explicar o churn, sem perda relevante de poder explicativo em relação ao modelo completo.


## 8. Curva ROC e Área Sob a Curva (AUC)

Utilizando a amostra de teste (não usada no ajuste do modelo), calculamos as probabilidades preditas de churn e construímos a Curva ROC, que avalia a capacidade do modelo de discriminar entre clientes que cancelam e clientes que não cancelam, para todos os pontos de corte possíveis.


In [ ]:
# Probabilidades preditas na amostra de teste
prob_teste <- predict(modelo_final, newdata = teste, type = "response")

# Curva ROC
roc_obj <- roc(teste$Churn, prob_teste)

plot(roc_obj, main = "Curva ROC - Modelo de Regressao Logistica", print.auc = TRUE, col = "darkblue")

auc(roc_obj)


### Escolha do ponto de corte

O ponto de corte (*threshold*) é escolhido de forma a maximizar simultaneamente sensibilidade e especificidade, utilizando o critério de Youden (índice J), disponível pela função `coords()` do pacote `pROC`. Essa escolha é preferível ao corte padrão de 0,5, especialmente em bases desbalanceadas como esta, em que o uso do corte de 0,5 tende a favorecer excessivamente a classe majoritária (não churn).


In [ ]:
ponto_corte <- coords(roc_obj, "best", best.method = "youden",
                       ret = c("threshold", "sensitivity", "specificity"))

ponto_corte


O ponto de corte obtido pelo índice de Youden é adotado na construção da matriz de confusão a seguir, por representar o melhor equilíbrio entre sensibilidade (capacidade de identificar corretamente os clientes que de fato cancelam) e especificidade (capacidade de identificar corretamente os clientes que permanecem), o que é especialmente relevante em um contexto de negócio no qual **deixar de identificar um cliente propenso ao churn** costuma ser mais custoso do que uma falsa identificação.


## 9. Matriz de Confusão e Medidas de Desempenho

Com o ponto de corte definido, classificamos os clientes da amostra de teste e construímos a matriz de confusão, calculando as principais medidas de desempenho.


In [ ]:
corte <- as.numeric(ponto_corte["threshold"])

pred_classe <- factor(ifelse(prob_teste >= corte, 1, 0), levels = c(0, 1))
classe_real <- factor(teste$Churn, levels = c(0, 1))

matriz_confusao <- confusionMatrix(pred_classe, classe_real, positive = "1")

matriz_confusao


In [ ]:
# Extracao individual das principais medidas de desempenho
acuracia     <- matriz_confusao$overall["Accuracy"]
sensibilidade <- matriz_confusao$byClass["Sensitivity"]
especificidade <- matriz_confusao$byClass["Specificity"]
vpp           <- matriz_confusao$byClass["Pos Pred Value"]
vpn           <- matriz_confusao$byClass["Neg Pred Value"]
f1            <- matriz_confusao$byClass["F1"]

data.frame(
  Metrica = c("Acuracia", "Sensibilidade", "Especificidade",
              "Valor Preditivo Positivo", "Valor Preditivo Negativo", "F1-score"),
  Valor = round(c(acuracia, sensibilidade, especificidade, vpp, vpn, f1), 3)
)


**Interpretação das medidas (a completar com os valores numéricos obtidos):**

- **Acurácia**: proporção geral de classificações corretas (clientes cujo status de churn foi corretamente previsto), considerando as duas classes.
- **Sensibilidade (recall)**: proporção de clientes que de fato cancelaram e que foram corretamente identificados pelo modelo — métrica central em um contexto de retenção de clientes, em que se deseja capturar o maior número possível de casos de risco.
- **Especificidade**: proporção de clientes que permaneceram e que foram corretamente identificados como tal.
- **Valor Preditivo Positivo (VPP)**: dentre os clientes classificados como propensos ao churn, a proporção que de fato cancelou.
- **Valor Preditivo Negativo (VPN)**: dentre os clientes classificados como não propensos ao churn, a proporção que de fato permaneceu.
- **F1-score**: média harmônica entre precisão (VPP) e sensibilidade, útil como resumo único de desempenho em bases desbalanceadas.

*Observação: os comentários acima devem ser ajustados pelo grupo de acordo com os valores numéricos efetivamente obtidos na execução do código.*


## 10. Conclusões

- O modelo de regressão logística ajustado permite estimar a probabilidade de cancelamento (*churn*) de um cliente a partir de suas características cadastrais, dos serviços contratados e das condições do contrato.
- A análise exploratória e a interpretação dos coeficientes (razões de chance) indicam que o **tipo de contrato**, o **tempo de permanência (tenure)**, o **método de pagamento** e a **mensalidade** figuram entre os fatores mais associados à chance de cancelamento, sendo contratos mensais e pagamento por cheque eletrônico associados a maior risco de churn, enquanto maior tempo de permanência está associado a menor risco.
- A capacidade discriminativa do modelo, avaliada pela AUC na amostra de teste, indica um desempenho preditivo (a ser classificado como bom/regular conforme o valor obtido) do modelo em distinguir clientes que cancelam de clientes que permanecem.
- A escolha do ponto de corte pelo índice de Youden, em vez do corte padrão de 0,5, foi importante dado o desbalanceamento da variável resposta, equilibrando sensibilidade e especificidade.

**Limitações:**

- O conjunto de dados é uma fotografia estática dos clientes, sem informação temporal detalhada sobre o histórico de uso dos serviços, o que limita a capacidade do modelo de capturar tendências de comportamento ao longo do tempo.
- O modelo logístico assume uma relação linear entre as covariáveis (na escala do logito) e a chance de churn, podendo não capturar relações não lineares ou interações relevantes entre variáveis.

**Possíveis extensões:**

- Testar modelos alternativos (árvores de decisão, random forest, gradient boosting) e comparar seu desempenho preditivo com o modelo logístico.
- Investigar interações entre covariáveis (por exemplo, entre tipo de contrato e método de pagamento).
- Aplicar técnicas de balanceamento de classes (undersampling, oversampling, SMOTE) e avaliar o impacto no desempenho do modelo.


## Referências

BLASTCHAR. *Telco Customer Churn*. Kaggle, 2018. Disponível em: <https://www.kaggle.com/datasets/blastchar/telco-customer-churn>.
